# 00 · 环境自检

跑一遍这个 notebook，每个格子都能出结果，就说明环境没问题，可以开始阶段 1 了。

**在 PyCharm 里**：右上角解释器选 `AI-Data/.venv/Scripts/python.exe`，然后点每个格子左边的 ▶，或者按 `Shift+Enter` 一格一格往下跑。

> 编程第一原则：**先确认环境，再写业务代码**。90% 的"代码跑不起来"其实是环境问题。

## 1. 解释器信息

重点看 `sys.executable` 是不是指向项目里的 `.venv`。
如果指向 `D:\develop\Python\python.exe`，说明你在用全局 Python，包管理迟早一团乱。

In [ ]:
import platform
import sys

print("Python  :", sys.version.split()[0])
print("解释器  :", sys.executable)
print("系统    :", platform.platform())

## 2. 库版本清单

阶段 1、2 需要的库都会列在这里。如果有 ❌，回到终端执行：

```powershell
pip install -r requirements\base.txt -i https://pypi.tuna.tsinghua.edu.cn/simple
```

In [ ]:
import importlib
import importlib.metadata as md

libs = [
    "numpy",
    "pandas",
    "matplotlib",
    "jupyterlab",
    "ipykernel",
    "seaborn",
    "plotly",
    "polars",
    "pyarrow",
]

for name in libs:
    try:
        importlib.import_module(name)
        print(f"{name:12s} OK   {md.version(name)}")
    except Exception as exc:
        print(f"{name:12s} 失败 {type(exc).__name__}: {exc}")

## 3. 第一个真实的小分析

用 numpy 造 200 个人的身高体重数据，用 pandas 算 BMI，用 matplotlib 画出来。

注意开头那两行 `rcParams`——**这是 Windows 上画中文图表的必备设置**，不设的话中文会变成方框。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Windows 中文字体设置：写在所有画图代码之前
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(42)  # 固定随机种子，保证结果可复现
df = pd.DataFrame(
    {
        "身高cm": rng.normal(170, 8, 200).round(1),
        "体重kg": rng.normal(65, 12, 200).round(1),
    }
)
df["BMI"] = (df["体重kg"] / (df["身高cm"] / 100) ** 2).round(2)

print(f"共 {len(df)} 行，{df.shape[1]} 列\n")
print(df.describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(df["身高cm"], df["体重kg"], alpha=0.6, s=25, color="#4C72B0")
axes[0].set_title("身高 vs 体重", fontsize=13)
axes[0].set_xlabel("身高 (cm)")
axes[0].set_ylabel("体重 (kg)")

axes[1].hist(df["BMI"], bins=20, color="#DD8452", edgecolor="white")
axes[1].axvline(
    df["BMI"].mean(), color="red", linestyle="--", label=f"均值 {df['BMI'].mean():.1f}"
)
axes[1].set_title("BMI 分布", fontsize=13)
axes[1].set_xlabel("BMI")
axes[1].set_ylabel("人数")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. 自检通过了吗

- 两张图都出来了，中文标题不是方框 → **环境 OK**
- 中文变成 `□□□` → 检查 `font.sans-serif`，执行 `plt.rcParams["font.sans-serif"]` 看当前值
- 报 `ModuleNotFoundError` → 解释器选错了，或者包没装上

## 5. 接下来

阶段 1 第一课是 numpy。动手之前，先自己想一个问题：

> 已经有 `list` 了，为什么还需要 numpy 数组？

带着这个问题往下学，比背 API 有用得多。